In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

DEEP LEARNING -- ASSIGNMENT 1
===

*Purpose of this assignment is to predict the ´charges´ column of the insurance.csv file through a Neural Network model.

Optuna, Optimizers, (???)

In [ ]:
#NECESSARY IMPORTS
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
 
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.nn.init as init
 
import optuna
from optuna.samplers import TPESampler
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

/home/ikergamboa/2025-26_DL_Assignments/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# SEED FOR REPRODUCIBILITY
# so we get the same results every time we run the notebook
SEED = 42

# !!!
# Set the random seed for both PyTorch and NumPy to ensure reproducibility of results.
torch.manual_seed(SEED)
np.random.seed(SEED)

# Check if an NVIDIA GPU (CUDA) is available. 
# If yes, we use 'cuda'; otherwise, we fall back to the 'cpu'.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Print the device choice so we know if the code is running on the GPU or CPU
print(f"Using device: {DEVICE}\n")

Using device: cpu



1. DATA PREPARATION AND UNDERSTANDING
---

Guides used (From Oficial PyTorch Documentation): [Data Tutorial](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) || [Data Loading Tutorial](https://docs.pytorch.org/tutorials/beginner/data_loading_tutorial.html)

In [3]:
# 0. DATASET LOADING
# We define a function with a default file path ("insurance.csv") 
# and specify that it returns a pandas DataFrame.
def load_data(path: str = "insurance.csv") -> pd.DataFrame:
    # read the CSV file into a DataFrame
    df = pd.read_csv(path)
    # Check the size
    print(f"  Shape        : {df.shape}")
    # Check column names
    print(f"  Columns      : {df.columns.tolist()}")
    # Check missing values
    print(f"  Missing vals : {df.isnull().sum().sum()}")
    # Statistical summary of the 'charges' column (our target variable)
    # This shows the mean, min, max, and percentiles to spot outliers early.
    print(f"  Charges stats:\n{df['charges'].describe().to_string()}\n")
    return df

# Execute the function and store the result in a variable named 'df'
df = load_data()

  Shape        : (1338, 7)
  Columns      : ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']
  Missing vals : 0
  Charges stats:
count     1338.000000
mean     13270.422265
std      12110.011237
min       1121.873900
25%       4740.287150
50%       9382.033000
75%      16639.912515
max      63770.428010



In [4]:
# 1.1 FEATURE ENGENEERING
# We use get_dummies to transform the 'region' column into multiple numeric columns.
# If 'region' has 4 categories (NE, NW, SE, SW), it will create 4 columns.
# 'drop_first=True' is used to avoid the "Dummy Variable Trap":
# It removes one column (e.g., 'region_northeast'). 
# If the remaining columns (NW, SE, SW) are all 0, the model knows the region MUST be NE.
# This prevents mathematical redundancy (multicollinearity) in your model.
data = pd.get_dummies(df, columns=['region'], drop_first=True)
# We manually map the strings to integers: Female becomes 0, Male becomes 1.
data['sex'] = data['sex'].map({'female': 0, 'male': 1})
# 'Binary Encoding' for smoking status:
# 'no' becomes 0 and 'yes' becomes 1.
data['smoker'] = data['smoker'].map({'no': 0, 'yes': 1})
# Display the first 5 rows of the new DataFrame to verify the changes.
data.head()


,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,False,False,True
1,18,1,33.770,1,0,1725.55230,False,True,False
2,28,1,33.000,3,0,4449.46200,False,True,False
3,33,1,22.705,0,0,21984.47061,True,False,False
4,32,1,28.880,0,0,3866.85520,True,False,False


In [5]:
# 1.2 PREPROCESSING (NORMALIZATION, STANDARDIZATION, ETC.)
# Define which numeric columns need to be scaled. It is not necessary to scale the one-hot encoded 'region' columns because they are already in a 0/1 format.
# We include 'charges' here because it's our target variable and we want it on a similar scale as the features.
cols_a_escalar = ['age', 'bmi', 'children', 'charges']

# Initialize the StandardScaler.
# This follows the formula: z = (x - mean) / standard_deviation
# It centers the data around 0 with a standard deviation of 1.
scaler = StandardScaler()

# fit_transform does two things:
# 1. 'fit' calculates the average (mean) and variance of each column.
# 2. 'transform' actually applies the math to change the numbers.
data[cols_a_escalar] = scaler.fit_transform(data[cols_a_escalar])
# Ensure all data in the DataFrame is a float (decimal number).
# This prevents errors during PyTorch tensor conversion later.
data = data.astype(float)
data.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,-1.438764,0.0,-0.453320,-0.908614,1.0,0.298584,0.0,0.0,1.0
1,-1.509965,1.0,0.509621,-0.078767,0.0,-0.953689,0.0,1.0,0.0
2,-0.797954,1.0,0.383307,1.580926,0.0,-0.728675,0.0,1.0,0.0
3,-0.441948,1.0,-1.305531,-0.908614,0.0,0.719843,1.0,0.0,0.0
4,-0.513149,1.0,-0.292556,-0.908614,0.0,-0.776802,1.0,0.0,0.0


In [6]:
# 1.3 DATA SPLITTING (TRAIN/VAL/TEST)
# X = every column except 'charges' (independent variable)
X = data.drop('charges', axis=1).values

# y = only the column 'charges' (dependent variable, what we want to predict)
y = data['charges'].values

# We split the data: 80% for training and 20% for testing.
# random_state=42 ensures the "random" split is the same every time you run the code.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# We print the number of rows in each set 
print(f"Muestras de entrenamiento: {len(X_train)}")
print(f"Muestras de prueba: {len(X_test)}")


# Convertimos a tensores de punto flotante (32 bits es el estándar en DL)
X_train = torch.tensor(X_train, dtype=torch.float32)
# .reshape(-1, 1) transforms 'y' from a flat list [1, 2, 3] 
# into a column vector [[1], [2], [3]], which PyTorch layers expect for regression.
y_train = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1) 

X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32).reshape(-1, 1)

# TensorDataset wraps the features and targets into a single object
train_ds = TensorDataset(X_train, y_train)
test_ds = TensorDataset(X_test, y_test)

# batch_size=32: Model studies 32 rows at a time before updating weights 
# data separated into 32 "groups" (batches)
batch_size = 32

# shuffle=True: Mixes the data so the model doesn't memorize 
# the specific order of the rows (improves generalization).train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

# No shuffle needed for testing; we just want to evaluate the fixed set.test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

Muestras de entrenamiento: 1070
Muestras de prueba: 268


2. NEURAL NETWORK DESIGN (Deep, Shallow...?)
---

In [ ]:
# 2.1 LAYERS AND UNITS
class InsuranceModel(nn.Module):
    def __init__(self, input_dim):
        # super () connects this class with the PyTorch nn.Module class, allowing us to use its built-in methods and properties.
        super(InsuranceModel, self).__init__()
        
        # LAYER 1: transforms the original columns (input_dim) into 64 new features (neurons).
        self.layer1 = nn.Linear(input_dim, 64)
        
        # LAYER 2: takes the 64 features from layer 1 and reduces them to 32 features.
        # this step allows the model to learn more complex patterns by combining the features from layer 1 in different ways.
        self.layer2 = nn.Linear(64, 32)

        # EXIT LAYER: takes the 32 features from layer 2 and reduces them to 1 output (the predicted cost).
        self.output_layer = nn.Linear(32, 1)
        
    def forward(self, x):
        # data goes into layer 1, then we apply the ReLU activation function to add non-linearity.
        x = self.activation(self.layer1(x))
        # data goes into layer 2, then we apply the ReLU activation function again to add non-linearity.
        x = self.activation(self.layer2(x))
        # data goes into the output layer, which gives us the final prediction (a single number for the insurance cost).
        x = self.output_layer(x)
        return x

In [ ]:
# 2.2 ACTIVATION FUNCTIONS
# Instanciamos el modelo definiendo las activaciones dentro
class InsuranceModel(nn.Module):
    def __init__(self, input_dim):
        super(InsuranceModel, self).__init__() # initializes the pytorch functionality for our model
        # defines the path of the data through the layers and activations in a single sequential block.
        self.net = nn.Sequential(
            # First layer: Multiplies the input features by weights and adds a bias to produce 64 new features.
            nn.Linear(input_dim, 64),
            # Applies the ReLU activation function to introduce non-linearity.
            nn.ReLU(),             
            # Second layer: Takes the 64 features from the first layer and transforms them into 32 new features.
            nn.Linear(64, 32),
            # Second ReLU activation to add non-linearity after the second layer.
            nn.ReLU(),   
            # Output layer: Takes the 32 features from the second layer and produces a single output (the predicted insurance cost).  
            # No activation function here because it's a regression problem; we want the output to be able to take any value, not just positive integers.         
            nn.Linear(32, 1)    
        )
        
    def forward(self, x):
        # The forward method defines how the input data flows through the network.
        return self.net(x)



InsuranceModel(
  (net): Sequential(
    (0): Linear(in_features=8, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)


3. TRAINING METHODOLOGY
---

In [ ]:
# 3.1 LOSS FUNCTION (Regression --> MSE?)

In [ ]:
# 3.2 OPTIMIZER (SGD, ADAM, ..., ?)

In [ ]:
# 3.3 INITIALIZATION (He Initialization)

# Obtains the number of input features from X_train (the number of columns) to initialize the model correctly.
input_size = X_train.shape[1]
# We create an instance of the InsuranceModel class, passing the number of input features as an argument.
model = InsuranceModel(input_size)

# He initialization (also known as Kaiming initialization) is a method for initializing the weights of neural networks, 
# especially those using ReLU activations.
def init_weights(m):
    # We check if the layer 'm' is an instance of nn.Linear, which means it's a fully connected layer.
    # We don't want to initialize activation functions or other types of layers,
    # only the linear layers that have weights and biases.
    if isinstance(m, nn.Linear):
        # We use Kaiming normal initialization for the weights of the layer 'm'.
        init.kaiming_normal_(m.weight, nonlinearity='relu')
        # We set the bias of the layer 'm' to a small constant value (0.01) to ensure that the 
        # neurons are active at the beginning of training.
        m.bias.data.fill_(0.01)

# Aplicarlo a tu modelo
model.apply(init_weights)   

print(model)

4. PERFORMANCE EVALUATION
---